<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/fleet-head-to-head.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · Fleet Racing Head-to-Head

Compare two schools across every fleet regatta they both attended in a
season: who beat whom, regatta by regatta, plus an overall record and a
chart of the rivalry over time.

This reproduces the "Fleet Racing Head-to-Head" page of the analytics
webapp (`FleetH2H.svelte`), which compares two schools' **regatta**
finishes (this notebook) or their **race-by-race** finishes (left as an
extension — see the closing note).

## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"

## Scrape the season's fleet results

`sailors=False` skips the per-sailor pages we don't need here (we're
comparing schools, not sailors), which roughly halves the fetches. The
first run scrapes the season (cached to disk); re-runs are instant.

In [ ]:
import scraper

# The full academic year: fall + spring.
SEASONS = ["f25", "s26"]

data = scraper.load(SEASONS, sailors=False, progress=True).fleet()
df = data.results_frame()
print(len(data.regattas), "fleet regattas |", len(df), "school results")
df.head()

## Pick two schools

Rather than hardcoding a rivalry that may not exist in this season's data,
find it: for every regatta, list the schools that attended, then count how
often each pair of schools showed up together. The pair with the most
shared regattas gives the richest head-to-head.

A school can enter more than one team in the same regatta (an A and a B
squad) — `results_frame()` has one row *per team*, not per school. We take
each school's **best-placed** team per regatta before counting or comparing,
which is also what the pairing table below does.

Swap `SCHOOL_A` / `SCHOOL_B` for any two slugs from `data.schools` to
compare a different pair.

In [ ]:
from collections import Counter
from itertools import combinations

# Best (lowest) place per school per regatta — collapses A/B squads down to
# one row per school, per the multi-team note above.
best = (
    df.sort_values("place")
    .groupby(["season", "regatta_slug", "school_slug", "school"], as_index=False)
    .first()
)

pair_counts = Counter()
for _, schools in best.groupby(["season", "regatta_slug"])["school_slug"]:
    for a, b in combinations(sorted(set(schools)), 2):
        pair_counts[(a, b)] += 1

SCHOOL_A, SCHOOL_B = pair_counts.most_common(1)[0][0]
assert SCHOOL_A in data.schools and SCHOOL_B in data.schools
print(
    f"Most shared regattas in {'+'.join(SEASONS)}: {SCHOOL_A} vs {SCHOOL_B} "
    f"({pair_counts[(SCHOOL_A, SCHOOL_B)]} regattas)"
)
pair_counts.most_common(5)

## Build the head-to-head table

Filter `best` (already one row per school per regatta) down to the two
schools, then pivot on `(season, regatta_slug)` so each shared regatta gets
one row with both schools' place and total score side by side. An inner
join naturally keeps only the regattas both schools attended.

In [ ]:
a = best[best["school_slug"] == SCHOOL_A]
b = best[best["school_slug"] == SCHOOL_B]

start_times = df[["season", "regatta_slug", "regatta_name", "start_time"]].drop_duplicates(
    ["season", "regatta_slug"]
)

h2h = (
    a.merge(b, on=["season", "regatta_slug"], suffixes=("_a", "_b"))
    .merge(start_times, on=["season", "regatta_slug"])
    .sort_values("start_time")
    .reset_index(drop=True)
)


def _winner(row):
    if row["place_a"] < row["place_b"]:
        return row["school_slug_a"]
    if row["place_b"] < row["place_a"]:
        return row["school_slug_b"]
    return "tie"


h2h["winner"] = h2h.apply(_winner, axis=1)

name_a, name_b = h2h["school_a"].iloc[0], h2h["school_b"].iloc[0]

h2h[
    [
        "start_time",
        "regatta_name",
        "school_a",
        "place_a",
        "total_a",
        "school_b",
        "place_b",
        "total_b",
        "winner",
    ]
]

## Overall record

In [ ]:
a_wins = int((h2h["winner"] == SCHOOL_A).sum())
b_wins = int((h2h["winner"] == SCHOOL_B).sum())
ties = int((h2h["winner"] == "tie").sum())
avg_gap = (h2h["place_b"] - h2h["place_a"]).abs().mean()

print(f"{name_a} vs {name_b}: {len(h2h)} shared regattas")
print(f"Record: {name_a} {a_wins} to {b_wins} {name_b}" + (f", {ties} ties" if ties else ""))
print(f"Average place gap: {avg_gap:.1f} places")

## Chart: place over the season

Both schools' finishing place at each shared regatta, ordered by date. The
y-axis is inverted so "up" always means a better finish.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(h2h["start_time"], h2h["place_a"], marker="o", label=name_a)
plt.plot(h2h["start_time"], h2h["place_b"], marker="o", label=name_b)
plt.gca().invert_yaxis()
plt.ylabel("Finishing place")
plt.xlabel("Regatta start date")
plt.title(f"{name_a} vs {name_b} \u2014 fleet head-to-head, {'+'.join(SEASONS)}")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

---
**Extending this:**
- Narrow to one school's whole schedule first with `data.school(slug)`
  (chainable, like every `Dataset` filter) before picking an opponent.
- Narrow to a single season by filtering the frames on the `season` column
  (the `(season, regatta_slug)` keys above stay collision-free, so annual
  regattas that repeat in both seasons never conflate).
- The webapp also has a **race-by-race** mode (every race both schools'
  boats sailed, not just the regatta-level result). That needs finer-grained
  data than `results_frame()`; `data.finishes_frame()` (per team·race place)
  gets you there — join each school's finishes on `(season, regatta_slug,
  division, race_num)` the same way this notebook joined on `(season,
  regatta_slug)`.